# AKE Lab 3 · Setup-Check (einmalig, mit Internet)

Lab 3 nutzt drei vortrainierte Modelle. MiniLM und die beiden Tokenizer liegen im Kursrepository unter `data/models/`.
Die **Gewichte von DistilBERT** (`model.safetensors`, 268 MB) sind zu groß für GitHub (Limit 100 MB pro Datei) und werden
**einmal** mit diesem Notebook vom Hugging-Face-Hub geladen – am besten zu Hause oder im Hochschul-WLAN **vor** dem Lab.
Im Lab selbst gibt es keinen Hub-Zugriff.

**Ablauf:** Zelle für Zelle mit *Shift + Enter* ausführen. Erwartung: alle Prüfungen ✅, Dauer je nach Leitung 1–5 Min.

**Kein Internet oder Hub gesperrt?** Die Datei `model.safetensors` vom Hochschul-Share (Ordner *Lab03 / models /
distilbert-base-uncased*) nach `data/models/distilbert-base-uncased/` im Repository kopieren und Zelle 2 erneut ausführen.
Es werden keine neuen Pakete installiert; alles Nötige ist seit `uv sync` aus dem ersten Lab vorhanden.

In [1]:
# 1 · Umgebung und Pfade
import os, sys, hashlib, importlib
from pathlib import Path

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden: Jupyter im Ordner des Repositorys starten und lab03/ oeffnen.")
DATA  = REPO_ROOT / "data"
ZIEL  = DATA / "models" / "distilbert-base-uncased"
DATEI = ZIEL / "model.safetensors"
REPO_ID, REVISION = "distilbert/distilbert-base-uncased", "12040accade4e8a0f71eabdb258fecc2e7e948be"   # feste Version
SHA256_OK = {"5e3f1108e3cb34ee048634875d8482665b65ac713291a7e32396fb18f6ff0063": "Hugging-Face-Hub, Revision 12040ac",
             "90295ed56fc68ca98b4d7011d1966e2149226ccb4226ae71b6935a723c665a31": "Spiegelkopie (Encoder identisch)"}

def check(name, ok, hint=""):
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {hint}")
    return ok

print("Python", sys.version.split()[0], "| REPO_ROOT:", REPO_ROOT)
for paket in ["torch", "transformers", "sentence_transformers", "sklearn", "umap", "huggingface_hub", "safetensors"]:
    try:
        m = importlib.import_module(paket)
        check(f"{paket} {getattr(m, '__version__', '')}", True)
    except ImportError:
        check(paket, False, "fehlt -> im Repository `uv sync` ausfuehren")
check("DistilBERT-Konfiguration und -Tokenizer im Repository", all((ZIEL / f).exists() for f in ["config.json", "tokenizer.json"]),
      "git pull im Repository")
check("MiniLM und GPT-2-Tokenizer im Repository",
      (DATA / "models" / "all-MiniLM-L6-v2" / "model.safetensors").exists() and (DATA / "models" / "gpt2-tokenizer" / "tokenizer.json").exists(),
      "git pull im Repository")

Python 3.12.10 | REPO_ROOT: /Users/stefan/Documents/02_PHWT/DS-Vorlesung/03_Git_Studierende
✅ torch 2.8.0
✅ transformers 4.57.6
✅ sentence_transformers 5.7.0
✅ sklearn 1.9.1
✅ umap 0.5.12
✅ huggingface_hub 0.36.2
✅ safetensors 0.8.0
✅ DistilBERT-Konfiguration und -Tokenizer im Repository
✅ MiniLM und GPT-2-Tokenizer im Repository


True

In [2]:
# 2 · DistilBERT-Gewichte laden (nur, wenn sie noch fehlen) und pruefen
if DATEI.exists():
    print(f"{DATEI.relative_to(REPO_ROOT)} ist bereits vorhanden ({DATEI.stat().st_size / 1e6:.0f} MB) – kein Download.")
else:
    from huggingface_hub import hf_hub_download
    print("Lade model.safetensors (268 MB) von huggingface.co/distilbert/distilbert-base-uncased ...")
    hf_hub_download(REPO_ID, "model.safetensors", revision=REVISION, local_dir=ZIEL)

h = hashlib.sha256()
with open(DATEI, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
check(f"Pruefsumme stimmt ({SHA256_OK.get(h.hexdigest(), 'unbekannt')})", h.hexdigest() in SHA256_OK,
      "Datei unvollstaendig oder falsch: loeschen und Zelle erneut ausfuehren")

data/models/distilbert-base-uncased/model.safetensors ist bereits vorhanden (265 MB) – kein Download.
✅ Pruefsumme stimmt (Spiegelkopie (Encoder identisch))


True

In [3]:
# 3 · Ladetest offline: genau so laedt das Lab die Modelle
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging
logging.set_verbosity_error()                       # die Warnung "newly initialized" ist hier noch uninteressant
from sentence_transformers import SentenceTransformer

tok = AutoTokenizer.from_pretrained(ZIEL)
model = AutoModelForSequenceClassification.from_pretrained(ZIEL, num_labels=2).eval()
enc = tok(["A wonderful film."], return_tensors="pt")
enc.pop("token_type_ids", None)                     # DistilBERT kennt keine Segment-IDs
with torch.inference_mode():
    logits = model(**enc).logits
n_param = f"{sum(p.numel() for p in model.parameters()):,}".replace(",", " ")
check(f"DistilBERT laedt: {n_param} Parameter, Logits {tuple(logits.shape)}",
      tuple(logits.shape) == (1, 2))
st = SentenceTransformer(str(DATA / "models" / "all-MiniLM-L6-v2"), device="cpu")
check(f"MiniLM laedt: Embedding-Laenge {st.encode(['A wonderful film.']).shape[1]}", True)
print("\nFertig. Lab 3 kann offline starten: lab03_starter.ipynb als lab03_<nachname>.ipynb kopieren.")

✅ DistilBERT laedt: 66 955 010 Parameter, Logits (1, 2)
✅ MiniLM laedt: Embedding-Laenge 384

Fertig. Lab 3 kann offline starten: lab03_starter.ipynb als lab03_<nachname>.ipynb kopieren.
